# 15 — ACSM: chemical speciation

**Theme:** what the aerosol mass is *made of*, rather than how large it is.

An Aerosol Chemical Speciation Monitor vaporises the non-refractory submicron
aerosol and measures the mass of each chemical species. Where a sizing
instrument tells you about particle diameter, this tells you about composition —
the two are complementary.

In [ ]:
import aerosoltools as at

acsm = at.load_simple_acsm_file("../../tests/data/Sample_ACSM.csv")

print("instrument:", acsm.instrument)
print("columns   :", list(acsm.data))
print("unit      :", acsm.unit)
print("samples   :", len(acsm.data))

## The species

Five species are reported, each as a mass concentration.

In [ ]:
for name in ["org", "sulfate", "nitrate", "ammonia", "chlorine"]:
    series = getattr(acsm, name)
    print(f"{name:9s} mean {series.mean():8.3f}   max {series.max():8.3f}")

Organics usually dominate submicron mass, with sulfate and nitrate next.
Chlorine is typically minor except near specific sources.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11, 4))
for name, colour in [("org", "tab:green"), ("sulfate", "tab:red"),
                     ("nitrate", "tab:blue"), ("ammonia", "tab:orange"),
                     ("chlorine", "tab:purple")]:
    ax.plot(acsm.time, getattr(acsm, name), label=name, lw=0.8)
ax.set_ylabel(f"Mass concentration [{acsm.unit}]")
ax.legend(ncol=5)
ax.set_title("Chemical composition over time")

## Composition as a whole

The relative contributions are usually more informative than the absolute
values, because they are what distinguishes one source or air mass from
another.

In [ ]:
species = ["org", "sulfate", "nitrate", "ammonia", "chlorine"]
means = {name: getattr(acsm, name).mean() for name in species}
total = sum(means.values())

for name, value in means.items():
    print(f"{name:9s} {value:8.3f}  ({100 * value / total:5.1f} %)")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.pie(means.values(), labels=list(means), autopct="%1.1f%%",
       colors=["tab:green", "tab:red", "tab:blue", "tab:orange", "tab:purple"])
ax.set_title("Mean submicron composition")

Negative values appear in ACSM data — they are the noise of a difference
measurement around a small signal, not physically negative mass. They matter
when averaging over short periods, where they can dominate; over longer periods
they average out.

## It is a non-particle class

The ACSM measures mass by species, not a particle number concentration, so
`total_concentration` deliberately raises.

In [ ]:
try:
    acsm.total_concentration
except AttributeError as err:
    print("AttributeError:", str(err)[:120])

The usual machinery still applies.

In [ ]:
midpoint = len(acsm.time) // 2
acsm.mark_activities({
    "First half":  [(str(acsm.time[0]), str(acsm.time[midpoint]))],
    "Second half": [(str(acsm.time[midpoint + 1]), str(acsm.time[-1]))],
})
acsm.summarize_activities()

Rebinning is often the first thing to do with an ACSM record: the instrument
samples every 30 seconds or so, and composition rarely changes that fast, so
averaging to a few minutes suppresses the noise without losing signal.

In [ ]:
hourly = acsm.timerebin("1h", inplace=False)

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(acsm.time, acsm.org, lw=0.5, alpha=0.4, label="as measured")
ax.plot(hourly.time, hourly.org, lw=2, label="hourly average")
ax.set_ylabel(f"Organics [{acsm.unit}]")
ax.legend()
ax.set_title("Rebinning suppresses the measurement noise")

---

That is the end of the current examples. The weather-station notebook (14) is
still to come.